In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
import networkx as nx
import json
import time
import pickle
import warnings
from collections import defaultdict
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

df         = pd.read_parquet("/kaggle/input/notebooks/anymansince2005/citation-graphs/arxiv_ml_graph_features.parquet")
df         = df.reset_index(drop=True)
embeddings = np.load("/kaggle/input/notebooks/anymansince2005/semantic-embedding/embeddings.npy")
faiss_index= faiss.read_index("/kaggle/input/notebooks/anymansince2005/semantic-embedding/faiss_index.bin")
with open("/kaggle/input/notebooks/anymansince2005/citation-graphs/citation_graph.pkl", "rb") as f:
    G = pickle.load(f)
model      = SentenceTransformer("all-MiniLM-L6-v2")

with open("/kaggle/input/notebooks/anymansince2005/citation-graphs/all_metrics.json") as f:
    all_metrics = json.load(f)

print(f"Loaded {len(df):,} papers")
print(f"Embeddings : {embeddings.shape}")
print(f"Graph      : {G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 110 papers
Embeddings : (110, 384)
Graph      : 110 nodes | 138 edges


In [3]:
class UserProfile:

    PIN_WEIGHT    = 3.0
    RECENCY_DECAY = 0.95   

    def __init__(self, user_id: str):
        self.user_id         = user_id
        self.read_indices    = []     
        self.pinned_indices  = set()
        self.category_counts = defaultdict(int)
        self.profile_vector  = None
        self.created_at      = pd.Timestamp.now()
        self.updated_at      = pd.Timestamp.now()

    def add_read(self, paper_idx: int, df: pd.DataFrame):
        if paper_idx not in self.read_indices:
            self.read_indices.append(paper_idx)
            for cat in df.iloc[paper_idx]["categories"]:
                self.category_counts[cat] += 1
            self.updated_at = pd.Timestamp.now()

    def pin_paper(self, paper_idx: int, df: pd.DataFrame):
        self.pinned_indices.add(paper_idx)
        if paper_idx not in self.read_indices:
            self.add_read(paper_idx, df)

    def build_profile_vector(self, embeddings: np.ndarray) -> np.ndarray:
        if not self.read_indices:
            return None

        vecs    = []
        weights = []
        n       = len(self.read_indices)

        for rank, idx in enumerate(self.read_indices):
            recency_w = self.RECENCY_DECAY ** (n - 1 - rank)
            pin_w     = self.PIN_WEIGHT if idx in self.pinned_indices else 1.0
            weight    = recency_w * pin_w

            vecs.append(embeddings[idx])
            weights.append(weight)

        weights = np.array(weights)
        weights = weights / weights.sum()              

        profile = np.average(vecs, axis=0, weights=weights)
        profile = profile / np.linalg.norm(profile)    

        self.profile_vector = profile.astype(np.float32)
        return self.profile_vector

    def top_categories(self, top_n: int = 3) -> list:
        return sorted(
            self.category_counts.items(),
            key=lambda x: -x[1]
        )[:top_n]

    def summary(self) -> dict:
        return {
            "user_id"       : self.user_id,
            "papers_read"   : len(self.read_indices),
            "papers_pinned" : len(self.pinned_indices),
            "top_categories": self.top_categories(3),
            "has_profile"   : self.profile_vector is not None,
        }


In [4]:
class ColdStartHandler:

    def __init__(self, df: pd.DataFrame, embeddings: np.ndarray, faiss_index):
        self.df          = df
        self.embeddings  = embeddings
        self.faiss_index = faiss_index
        self._popularity_cache = None

    def recommend_from_seeds(
        self,
        seed_titles : list[str],
        top_n       : int = 10,
    ) -> pd.DataFrame:
        seed_indices = []
        for title in seed_titles:
            match = self.df[
                self.df["title"].str.lower().str.contains(
                    title.lower(), regex=False
                )
            ]
            if not match.empty:
                seed_indices.append(match.index[0])

        if not seed_indices:
            print("  No seed papers matched — falling back to popularity")
            return self.recommend_popular(top_n)

        seed_vecs   = self.embeddings[seed_indices].mean(axis=0, keepdims=True)
        seed_vecs   = (seed_vecs / np.linalg.norm(seed_vecs)).astype(np.float32)

        k_fetch      = top_n + len(seed_indices)
        scores, idxs = self.faiss_index.search(seed_vecs, k_fetch)

        results = [
            (int(i), float(s))
            for i, s in zip(idxs[0], scores[0])
            if int(i) not in seed_indices
        ][:top_n]

        rec_df = self.df.iloc[[r[0] for r in results]][
            ["paper_id", "title", "categories", "date", "graph_pagerank"]
        ].copy()
        rec_df["score"]  = [round(r[1], 4) for r in results]
        rec_df["source"] = "seed_papers"
        return rec_df.reset_index(drop=True)

    def recommend_by_categories(
        self,
        selected_cats : list[str],
        top_n         : int = 10,
    ) -> pd.DataFrame:
        mask = self.df["categories"].apply(
            lambda cats: bool(set(cats) & set(selected_cats))
        )
        subset = self.df[mask].copy()

        if subset.empty:
            return self.recommend_popular(top_n)

        subset = subset.sort_values("graph_pagerank", ascending=False)

        rec_df = subset.head(top_n)[
            ["paper_id", "title", "categories", "date", "graph_pagerank"]
        ].copy()
        rec_df["score"]  = rec_df["graph_pagerank"].round(6)
        rec_df["source"] = "category_popular"
        return rec_df.reset_index(drop=True)

    def recommend_popular(self, top_n: int = 10) -> pd.DataFrame:
        if self._popularity_cache is None:
            self._popularity_cache = (
                self.df.sort_values("graph_pagerank", ascending=False)
                .head(100)
            )

        rec_df = self._popularity_cache.head(top_n)[
            ["paper_id", "title", "categories", "date", "graph_pagerank"]
        ].copy()
        rec_df["score"]  = rec_df["graph_pagerank"].round(6)
        rec_df["source"] = "popularity_fallback"
        return rec_df.reset_index(drop=True)

    def recommend(
        self,
        seed_titles   : list[str] = None,
        categories    : list[str] = None,
        top_n         : int = 10,
    ) -> pd.DataFrame:
        if seed_titles:
            return self.recommend_from_seeds(seed_titles, top_n)
        if categories:
            return self.recommend_by_categories(categories, top_n)
        return self.recommend_popular(top_n)


In [5]:
def recommend_for_user(
    profile         : UserProfile,
    embeddings      : np.ndarray,
    faiss_index,
    df              : pd.DataFrame,
    G               : nx.DiGraph,
    top_n           : int = 10,
    alpha           : float = 0.70,
    beta            : float = 0.20,
    exclude_read    : bool  = True,
) -> pd.DataFrame:
    profile_vec = profile.build_profile_vector(embeddings)
    if profile_vec is None:
        raise ValueError("User has no reading history — use ColdStartHandler")

    query_vec        = profile_vec.reshape(1, -1)
    k_fetch          = min(top_n * 10, len(df))
    sem_scores, idxs = faiss_index.search(query_vec, k_fetch)

    rows = []
    for i, s in zip(idxs[0], sem_scores[0]):
        idx = int(i)
        if exclude_read and idx in profile.read_indices:
            continue

        graph   = float(df.iloc[idx]["graph_score"])
        blended = alpha * float(s) + beta * graph

        rows.append({
            "cand_idx"      : idx,
            "semantic_score": round(float(s), 4),
            "graph_score"   : round(graph,    4),
            "blended_score" : round(blended,  4),
        })

    result_df = (
        pd.DataFrame(rows)
        .sort_values("blended_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    meta = df.iloc[result_df["cand_idx"].tolist()][
        ["paper_id", "title", "categories", "date"]
    ].reset_index(drop=True)

    return pd.concat([meta, result_df.drop(columns="cand_idx")], axis=1)

def more_like_this(
    paper_idx   : int,
    embeddings  : np.ndarray,
    faiss_index,
    df          : pd.DataFrame,
    top_n       : int = 10,
    exclude_idxs: list = None,
) -> pd.DataFrame:
    exclude = set(exclude_idxs or [])
    exclude.add(paper_idx)

    query_vec        = embeddings[paper_idx].reshape(1, -1).astype(np.float32)
    k_fetch          = top_n + len(exclude) + 10
    sem_scores, idxs = faiss_index.search(query_vec, k_fetch)

    results = [
        (int(i), float(s))
        for i, s in zip(idxs[0], sem_scores[0])
        if int(i) not in exclude
    ][:top_n]

    rec_df = df.iloc[[r[0] for r in results]][
        ["paper_id", "title", "categories", "date"]
    ].copy()
    rec_df["similarity_score"] = [round(r[1], 4) for r in results]
    rec_df["source"]           = "more_like_this"
    return rec_df.reset_index(drop=True)

In [6]:
def simulate_user_session(
    user_id         : str,
    initial_reads   : list[int],
    pinned          : list[int],
    df              : pd.DataFrame,
    embeddings      : np.ndarray,
    faiss_index,
    G               : nx.DiGraph,
    cold_start      : ColdStartHandler,
    top_n           : int = 10,
) -> dict:
    profile = UserProfile(user_id)
    results = {}

    print(f"\n{'─'*60}")
    print(f"USER: {user_id}")
    print(f"{'─'*60}")
    print("\n[Stage 0] New user — cold start (popularity fallback)")
    cs_recs = cold_start.recommend(top_n=top_n)
    results["cold_start"] = cs_recs
    print(cs_recs[["title", "score", "source"]].head(5).to_string(index=False))

    first_read = initial_reads[0]
    profile.add_read(first_read, df)
    print(f"\n[Stage 1] Read: '{df.iloc[first_read]['title'][:70]}'")
    cs_seed = cold_start.recommend(
        seed_titles=[df.iloc[first_read]["title"]],
        top_n=top_n
    )
    results["after_first_read"] = cs_seed
    print(cs_seed[["title", "score", "source"]].head(5).to_string(index=False))

    for idx in initial_reads[1:]:
        profile.add_read(idx, df)

    print(f"\n[Stage 2] Read {len(profile.read_indices)} papers — warm profile active")
    warm_recs = recommend_for_user(
        profile, embeddings, faiss_index, df, G, top_n=top_n
    )
    results["warm_recs"] = warm_recs
    print(warm_recs[["title", "semantic_score", "blended_score"]].head(5).to_string(index=False))

    for idx in pinned:
        profile.pin_paper(idx, df)

    print(f"\n[Stage 3] Pinned: '{df.iloc[pinned[0]]['title'][:70]}'")
    mlt_recs = more_like_this(
        pinned[0], embeddings, faiss_index, df,
        top_n=top_n, exclude_idxs=profile.read_indices
    )
    results["more_like_this"] = mlt_recs
    print(mlt_recs[["title", "similarity_score"]].head(5).to_string(index=False))

    print(f"\n[Stage 4] Updated profile after pin")
    updated_recs = recommend_for_user(
        profile, embeddings, faiss_index, df, G, top_n=top_n
    )
    results["updated_warm"] = updated_recs
    print(updated_recs[["title", "semantic_score", "blended_score"]].head(5).to_string(index=False))

    print(f"\nProfile summary: {profile.summary()}")
    results["profile"] = profile

    return results


In [7]:
cold_start = ColdStartHandler(df, embeddings, faiss_index)

session = simulate_user_session(
    user_id       = "researcher_001",
    initial_reads = [0, 5, 18, 42, 77],
    pinned        = [42],
    df            = df,
    embeddings    = embeddings,
    faiss_index   = faiss_index,
    G             = G,
    cold_start    = cold_start,
    top_n         = 10,
)

print("\n" + "=" * 60)
print("SIMULATING MULTIPLE USER PROFILES")
print("=" * 60)

np.random.seed(42)

USER_CONFIGS = [
    {
        "user_id"      : "nlp_researcher",
        "categories"   : ["cs.CL", "cs.LG"],
        "reads"        : list(np.random.choice(
            df[df["categories"].apply(lambda c: "cs.CL" in c)].index, 8, replace=False
        )),
        "pins"         : [],
    },
    {
        "user_id"      : "cv_researcher",
        "categories"   : ["cs.CV", "cs.LG"],
        "reads"        : list(np.random.choice(
            df[df["categories"].apply(lambda c: "cs.CV" in c)].index, 8, replace=False
        )),
        "pins"         : [],
    },
    {
        "user_id"      : "new_user",
        "categories"   : [],
        "reads"        : [],
        "pins"         : [],
    },
]

profiles = {}

for cfg in USER_CONFIGS:
    user    = UserProfile(cfg["user_id"])
    for idx in cfg["reads"]:
        user.add_read(int(idx), df)
    for idx in cfg["pins"]:
        user.pin_paper(int(idx), df)

    profiles[cfg["user_id"]] = user
    print(f"\n{cfg['user_id']:20} — {user.summary()}")

    if user.read_indices:
        recs = recommend_for_user(user, embeddings, faiss_index, df, G, top_n=5)
        print(f"  Top rec: {recs.iloc[0]['title'][:80]}")
    else:
        recs = cold_start.recommend(categories=cfg["categories"], top_n=5)
        print(f"  Cold start top rec: {recs.iloc[0]['title'][:80]}")



────────────────────────────────────────────────────────────
USER: researcher_001
────────────────────────────────────────────────────────────

[Stage 0] New user — cold start (popularity fallback)
                                                                                              title    score              source
                                    On complexity of optimized crossover for binary representations 0.048005 popularity_fallback
                               A Novel Model of Working Set Selection for SMO Decomposition Methods 0.029403 popularity_fallback
                                                            Mixed membership stochastic blockmodels 0.029327 popularity_fallback
                                                                              Compressed Regression 0.027335 popularity_fallback
Condition Monitoring of HV Bushings in the Presence of Missing Data\n  Using Evolutionary Computing 0.025948 popularity_fallback

[Stage 1] Read: 'Intellige

In [8]:
def evaluate_cold_start(df, cold_start, sample_size=100):
    indices    = np.random.choice(len(df), size=sample_size, replace=False)
    precisions = []

    for idx in indices:
        query_cats = df.iloc[idx]["categories"]
        seed_title = df.iloc[idx]["title"]

        recs   = cold_start.recommend(seed_titles=[seed_title], top_n=10)
        hits   = sum(
            1 for cats in recs["categories"].tolist()
            if set(cats) & set(query_cats)
        )
        precisions.append(hits / 10)

    return {
        "cold_start_precision@10": round(np.mean(precisions), 4),
        "std"                    : round(np.std(precisions), 4),
        "sample_size"            : sample_size,
    }


print("\n" + "=" * 60)
print("COLD START EVALUATION")
print("=" * 60)

np.random.seed(42)
cs_metrics = evaluate_cold_start(df, cold_start, sample_size=100)
print(f"\n  Cold start precision@10 : {cs_metrics['cold_start_precision@10']}")
print(f"  Std                     : {cs_metrics['std']}")

all_metrics["cold_start"] = cs_metrics


COLD START EVALUATION

  Cold start precision@10 : 0.547
  Std                     : 0.2492


In [9]:
print("\n" + "=" * 60)
print("SAVING ARTIFACTS")
print("=" * 60)

with open("/kaggle/working/user_profiles.pkl", "wb") as f:
    pickle.dump(profiles, f)
print("Saved: user_profiles.pkl")

df.to_parquet("/kaggle/working/arxiv_ml_final.parquet", index=False)
print("Saved: arxiv_ml_final.parquet")

with open("/kaggle/working/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print("Saved: all_metrics.json")


SAVING ARTIFACTS
Saved: user_profiles.pkl
Saved: arxiv_ml_final.parquet
Saved: all_metrics.json
